### PyTorch Lightning basic classification model ###
A more advanced implementation with TensorBoard logging

In [1]:
import sys
import os
import copy
import pandas as pd
import numpy as np
import json
from pathlib import Path
import albumentations as alb
from matplotlib import pyplot as plt

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights

# Lightning module
from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateFinder, LearningRateMonitor

# Appearance of the Notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Import this module with autoreload
%load_ext autoreload
%autoreload 2

import computervision as cv
from computervision.fileutils import FileOP
from computervision.imageproc import ImageData, is_image
from computervision.transformations import AugmentationTransform
from computervision.datasets import DatasetFromDF
from computervision.inference import get_gpu_info
from computervision.models.lightningmodel import ToothModel, FineTuneLearningRateFinder

# Print version info
print(f'Package version: {cv.__version__}')
print(f'Authors:         {cv.__authors__}')
print(f'Python version:  {sys.version}')

Package version: v0.0.2
Authors:         The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [9]:
# Path settings 
# Main data directory (defined as environment variable in docker-compose.yml)
dataset_name = 'dentex_classification'
data_dir = os.path.join(os.environ.get('DATA'), 'dentex', dataset_name)

# Download directory (change as needed)
model_dir = os.path.join(data_dir, 'model')

# This image directory is where the xrays are in the archive, so should be left as-is
raw_image_dir = os.path.join(data_dir, 'quadrant-enumeration-disease', 'xrays')
image_dir = os.path.join(raw_image_dir, 'crop')

data_file_name = 'dentex_disease_datasplit.parquet'
data_file = os.path.join(data_dir, data_file_name)

### Load the annotations ###

In [12]:
display(df.head())

,image_id,file_name,image_number,file_path,quadrant,position,label,cl,area,bbox,box_name,annotations,box_file,im_width,im_height,dataset,category
0,272,train_191.png,191,/app/data_model/dentex/dentex_classification/q...,1,8,Impacted,0,39275,"[677.0, 446.0, 276.0, 207.0]",train_191_1435_1_8,9,/app/data_model/dentex/dentex_classification/q...,276,207,train,2
1,600,train_418.png,418,/app/data_model/dentex/dentex_classification/q...,1,7,Caries,1,34312,"[790.2912621359224, 372.81553398058253, 141.74...",train_418_3035_1_7,9,/app/data_model/dentex/dentex_classification/q...,142,311,train,0
2,202,train_391.png,391,/app/data_model/dentex/dentex_classification/q...,3,6,Caries,1,49152,"[1892.0, 767.0, 286.0, 316.0]",train_391_1068_3_6,17,/app/data_model/dentex/dentex_classification/q...,286,316,train,0
3,535,train_659.png,659,/app/data_model/dentex/dentex_classification/q...,4,8,Caries,1,42396,"[753.0, 611.0, 269.0, 277.0]",train_659_2767_4_8,1,/app/data_model/dentex/dentex_classification/q...,269,277,train,0
4,632,train_587.png,587,/app/data_model/dentex/dentex_classification/q...,4,6,Caries,1,45572,"[798.9655172413794, 655.2068965517242, 170.689...",train_587_3155_4_6,9,/app/data_model/dentex/dentex_classification/q...,171,324,train,0


In [13]:
annotations_file_name = 'dentex_disease_datasplit.parquet'
annotations_file = os.path.join(data_dir, annotations_file_name)
df = pd.read_parquet(annotations_file)

file_col = 'file_path'
bbox_col= 'bbox'
label_col = 'label'
dset_col = 'dataset'

labels = sorted(list(df[label_col].unique()))
label2id = dict(zip(labels, range(len(labels))))
id2label = {category_id: label for label, category_id in label2id.items()}
display(id2label)

# Now we can add a category id to the data frame
df = df.assign(category=df[label_col].apply(lambda label: label2id.get(label)))
display(df.head())

# Check the images
file_list = [os.path.join(image_dir, file_name) for file_name in df[file_col].unique()]
checked = [is_image(file) for file in file_list]
assert len(file_list) == sum(checked), f'WARNING: Could not open all {len(file_list)} images at: {image_dir}'
print(f'Image directory:        {image_dir}')
print(f'Total number of images: {len(file_list)}')
print(f'Annotations:            {df.shape[0]}')

{0: 'Caries', 1: 'Deep Caries', 2: 'Impacted', 3: 'Periapical Lesion'}

,image_id,file_name,image_number,file_path,quadrant,position,label,cl,area,bbox,box_name,annotations,box_file,im_width,im_height,dataset,category
0,272,train_191.png,191,/app/data_model/dentex/dentex_classification/q...,1,8,Impacted,0,39275,"[677.0, 446.0, 276.0, 207.0]",train_191_1435_1_8,9,/app/data_model/dentex/dentex_classification/q...,276,207,train,2
1,600,train_418.png,418,/app/data_model/dentex/dentex_classification/q...,1,7,Caries,1,34312,"[790.2912621359224, 372.81553398058253, 141.74...",train_418_3035_1_7,9,/app/data_model/dentex/dentex_classification/q...,142,311,train,0
2,202,train_391.png,391,/app/data_model/dentex/dentex_classification/q...,3,6,Caries,1,49152,"[1892.0, 767.0, 286.0, 316.0]",train_391_1068_3_6,17,/app/data_model/dentex/dentex_classification/q...,286,316,train,0
3,535,train_659.png,659,/app/data_model/dentex/dentex_classification/q...,4,8,Caries,1,42396,"[753.0, 611.0, 269.0, 277.0]",train_659_2767_4_8,1,/app/data_model/dentex/dentex_classification/q...,269,277,train,0
4,632,train_587.png,587,/app/data_model/dentex/dentex_classification/q...,4,6,Caries,1,45572,"[798.9655172413794, 655.2068965517242, 170.689...",train_587_3155_4_6,9,/app/data_model/dentex/dentex_classification/q...,171,324,train,0


Image directory:        /app/data_model/dentex/dentex_classification/quadrant-enumeration-disease/xrays/crop
Total number of images: 678
Annotations:            3529


### Image augmentations for training and validation/testing ###

In [14]:
# Initial scaling and padding for the bigger dimension
max_image_size = 550

# Model input size
im_width, im_height = 224, 224
train_transforms = AugmentationTransform(im_width=im_width, im_height=im_height).\
                get_transforms(name='train_transform')

# Resize and then normalize 
# with ImageNet mean and standard deviation for ResNet50
image_net_mean = [0.485, 0.456, 0.406]
image_net_std = [0.229, 0.224, 0.225]

# This transform is essential and needs to be applies for both training and validation
resize_and_normalize = [alb.Resize(width=im_width, height=im_height),
                        alb.Normalize(mean=image_net_mean, std=image_net_std)]
train_transforms.extend(resize_and_normalize)
train_transform = alb.Compose(train_transforms)

# However, for validation and testing, we don't want the augmentations, 
# so we just resize and normalize the data
test_transform = alb.Compose(resize_and_normalize)

### Datasets from the annotations data frame ###

In [15]:
# Create the data sets from the data frame
train_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'train'],
                              image_dir=image_dir,
                              file_name_col=file_col,
                              label_id_col='category',
                              max_image_size=max_image_size,
                              transform=train_transform,
                              validate=True)


val_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'val'],
                            image_dir=image_dir,
                            file_name_col=file_col,
                            label_id_col='category',
                            max_image_size=max_image_size,
                            transform=test_transform,
                            validate=True)

test_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'test'],
                             image_dir=image_dir,
                             file_name_col=file_col,
                             label_id_col='category',
                             max_image_size=max_image_size,
                             transform=test_transform,
                             validate=True)


print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

3289
120
120


### Model parameters and rate scheduling ###

In [16]:
device_number = 0
device, device_str = get_gpu_info(device_number=device_number)

Current device:    cuda:0


In [17]:
# Model parameters
model_name = 'dentexmodel'
model_version = 1

model_name_dir = os.path.join(model_dir, model_name)
checkpoint_dir = os.path.join(model_name_dir, f'{model_name}_{model_version}')
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

model_info = {'model_version': 1,
              'device_number': device_number,
              'project_version': cv.__version__,
              'model_name': model_name,
              'image_dir': image_dir,
              'model_dir': checkpoint_dir,
              'im_width': im_width,
              'im_height': im_height,
              'max_image_size': max_image_size}

training_args = {'max_epochs': 5,
                 'num_classes': 4,
                 'num_workers': 2,
                 'batch_size': 16,
                 'initial_lr': 1.0e-3,
                 'check_val_every_n_epoch': 1,
                 'checkpoint_very_n_epoch': 2,
                 'save_top_k': 3}

# Save the model parameters
parameters = {'model_info': model_info,
              'id2label': id2label,
              'training_args': training_args}

json_file = os.path.join(checkpoint_dir, f'{model_name}.json')
with open(json_file, 'w') as f:
    json.dump(parameters, f, indent=4)

### Create the model ###

In [18]:
model = ToothModel(train_dataset=train_dataset,
                   val_dataset=val_dataset,
                   test_dataset=test_dataset,
                   batch_size=training_args.get('batch_size'),
                   num_classes=training_args.get('num_classes'),
                   num_workers=training_args.get('num_workers'),
                   lr=training_args.get('initial_lr'))

### Logger and checkpoints ###

In [19]:
# Directory to save checkpoints and logs
chk_callback = ModelCheckpoint(dirpath=checkpoint_dir,
                               filename='model-{epoch}',
                               monitor='val_loss',
                               mode='min',
                               save_last=True,
                               every_n_epochs=training_args.get('checkpoint_every_n_epoch'),
                               save_on_train_epoch_end=True,
                               save_top_k=training_args.get('save_top_k'))

# Setup logger
logger = TensorBoardLogger(save_dir=model_name_dir,
                           name=model_name,
                           version=model_version)

### Learning rate fine tuning ###

In [20]:
lr_finder = FineTuneLearningRateFinder(milestones=(5, 10), 
                                       min_lr=1.0e-8,  
                                       max_lr=0.01, 
                                       num_training_steps=100,
                                       mode='exponential',
                                       early_stop_threshold=None,
                                       update_attr=True)

lr_starter = LearningRateFinder(min_lr=1.0e-8,  
                                max_lr=0.01, 
                                num_training_steps=300,
                                mode='exponential',
                                early_stop_threshold=None,
                                update_attr=True)

lr_monitor = LearningRateMonitor(logging_interval='epoch',
                                 log_momentum=True)

In [21]:
tr = Trainer(max_epochs=training_args.get('max_epochs'),
             default_root_dir=checkpoint_dir,
             callbacks=[chk_callback, lr_monitor],
             logger=logger,
             check_val_every_n_epoch=training_args.get('check_val_every_n_epoch'))

tr.fit(model)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:751: Checkpoint directory /app/data_model/dentex/dentex_classification/model/dentexmodel/dentexmodel_1 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet           | 24.6 M | train
1 | criterion | CrossEntropyLoss | 0      | train
2 | metrics   | ModuleDict       | 0      | train
-------------------------------------------------------
24.6 M    Trainable params
0         Non-trainable params
24.6 M    Total params
98.237    Total estimated model params size (MB)
161       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
